In [6]:
# cluster_and_rename.py

import json
import os
import numpy as np
import hdbscan
import pandas as pd
from sentence_transformers import SentenceTransformer
import requests

# =========================
# DeepSeek API配置
# =========================
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_URL = "https://api.deepseek.com/v1/chat/completions"
MODEL_NAME = "deepseek-chat"

INPUT_FILE = "test.json"
OUTPUT_FILE = "normalized_triples.json"
CLUSTER_CSV = "cluster_results.csv"

# 本地embedding模型
EMBEDDING_MODEL = "D:\\models\\sentence-transformers\\bge-large-zh"


# =========================
# LLM命名函数
# =========================
def call_llm(prompt):
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": "你是一个专业的事件抽象命名专家。"},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0
    }

    response = requests.post(DEEPSEEK_URL, headers=headers, json=payload)
    return response.json()["choices"][0]["message"]["content"].strip()


# =========================
# 聚类
# =========================
def cluster_events(events):
    model = SentenceTransformer(EMBEDDING_MODEL)
    embeddings = model.encode(events, show_progress_bar=True)

    print("向量计算完成，开始HDBSCAN聚类...")

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=4,
        metric='euclidean'
    )

    labels = clusterer.fit_predict(embeddings)
    return labels


# =========================
# 主函数
# =========================
def main():

    # 读取三元组
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        triples = json.load(f)

    # 收集唯一事件
    events = []
    for t in triples:
        events.append(t["cause"])
        events.append(t["effect"])

    unique_events = list(set(events))

    print(f"共 {len(unique_events)} 个唯一事件，开始聚类...")

    labels = cluster_events(unique_events)

    # 组织cluster
    cluster_dict = {}
    for event, label in zip(unique_events, labels):
        if label == -1:
            continue
        cluster_dict.setdefault(label, []).append(event)

    print(f"识别出 {len(cluster_dict)} 个有效聚类")

    # LLM命名
    cluster_name_map = {}

    for label, event_list in cluster_dict.items():
        prompt = f"""
以下事件表达语义相似，请为它们起一个统一抽象名称：

{event_list}

只输出一个简洁名称。
"""
        name = call_llm(prompt)
        cluster_name_map[label] = name
        print(f"Cluster {label} → {name}")

    # 建立映射
    event_to_cluster = {}

    for event, label in zip(unique_events, labels):
        if label != -1:
            event_to_cluster[event] = cluster_name_map[label]
        else:
            event_to_cluster[event] = event  # 噪声点保留原始表达

    # 生成标准化三元组
    normalized_triples = []

    for t in triples:
        normalized_triples.append({
            "cause": event_to_cluster[t["cause"]],
            "relation": t["relation"],
            "effect": event_to_cluster[t["effect"]]
        })

    # 保存标准化JSON
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(normalized_triples, f, ensure_ascii=False, indent=2)

    print(f"标准化三元组已保存至 {OUTPUT_FILE}")

    # =========================
    # 新增：保存聚类结果为CSV
    # =========================
    rows = []

    for event, label in zip(unique_events, labels):
        rows.append({
            "event_text": event,
            "cluster_id": label,
            "cluster_name": cluster_name_map.get(label, "noise")
        })

    df = pd.DataFrame(rows)
    df.to_csv(CLUSTER_CSV, index=False, encoding="utf-8-sig")

    print(f"聚类结果已保存至 {CLUSTER_CSV}")


if __name__ == "__main__":
    main()

共 26 个唯一事件，开始聚类...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.35it/s]
c:\Users\yc\anaconda3\envs\bertopic_env\lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yc\anaconda3\envs\bertopic_env\lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


向量计算完成，开始HDBSCAN聚类...
识别出 2 个有效聚类
Cluster 0 → 环境污染事件
Cluster 1 → 舆情波动
标准化三元组已保存至 normalized_triples.json
聚类结果已保存至 cluster_results.csv
